## Задание 1 (8 баллов)

Дообучите три любые разные модели с использованием DPO. Можете использовать другие датасеты с chosen/rejected структурой. Попробуйте обучать модель на как можно большем количестве примеров, которые позволит вам colab. Можете попробовать менять параметры обучения тоже.

Для каждой дообученой модели подготовьте небольшой репорт, в котором вы сравниваете исходные предсказания и предсказания после дообучения. Найдите несколько закономерностей глазами, а также посчитайте простые статистики (длина, количество уникальных символов и тп)

В конце проверьте есть ли пересечения паттернов между моделями.

## Задание 2 (доп 2 балла)

Чтобы получить два доп балла вам нужно показать, что модель после дообучения стала действительно лучше (приведите как минимум три аргумента с примерами!).
Если так не получается сразу, то поработайте над обучающими параметрами (проведите больше экспериментов)

РЕПОРТЫ ПО МОДЕЛЯМ
---

Общие выводы - DPO пока не успел внести существенные смысловые изменения из-за короткого цикла обучения (но увеличив кол-во шагов до 300 и 500 я тоже не увидела изменений -> нужно менять параметры, а они уже удлиняют срок обучения до 12+ часов), возможно, стоило взять модели попроще, но было интересно попробовать на каких-то популярных кандидатах.

Данные:
1) датасет: ultrafeedback_binarized
2) train: 5000
3) eval: 50

Репорты по моделям:
---
> Qwen/Qwen2.5-1.5B-Instruct

- в примерах 4,5 обрывается генерация (могу предположить, что это из-за ограничений длины вывода)
- ответы идентичны, не изенений в тоне, длине ответа (DPO не изменило поведение модели)

---


> TinyLlama/TinyLlama-1.1B-Chat-v1.0

- ответы тоже идентичные
- присутствуют проблемы с галлюцинациями (1 и 2 пример, watershed ≠ синоним water и не отвечает на условие и «Lost City of the Monkey God» находится в Гондурасе, а не в Мексике)

---

> google/gemma-2b-it

- ответы тоже идентичные
- Ответы корректные, грамотно сформулированы, структурированы списками, объяснения логичны





In [2]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [3]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig
from tqdm import tqdm
import numpy as np

In [32]:
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

TRAIN_SIZE = 5000
EVAL_SIZE = 50

dataset_train = dataset.select(range(TRAIN_SIZE))
dataset_eval  = dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + EVAL_SIZE))

print(f"Train: {len(dataset_train)}, Eval: {len(dataset_eval)}")

Train: 5000, Eval: 50


In [16]:
models = [
    "Qwen/Qwen2.5-3B-Instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "google/gemma-2b-it"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_args = DPOConfig(
    output_dir="dpo_output",
    beta=0.1,
    loss_type="sigmoid",
    max_length=128,
    num_train_epochs=1,
    max_steps=150,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    logging_steps=20,
    report_to="none",
    bf16=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
def train_single_model(model_name, adapter_name):
    print(f"\n===== TRAINING {model_name} =====")

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    model.config.use_cache = False

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    trainer = DPOTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset_train,
        eval_dataset=dataset_eval,
        processing_class=tokenizer,
        peft_config=peft_config,
    )

    trainer.train()

    trainer.model.save_pretrained(adapter_name)

    del model
    del trainer
    torch.cuda.empty_cache()

    print(f"Saved to {adapter_name}")

In [18]:
torch.cuda.empty_cache()

In [19]:
train_single_model("Qwen/Qwen2.5-1.5B-Instruct","adapter_qwen")


===== TRAINING Qwen/Qwen2.5-1.5B-Instruct =====


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,0.693534
40,0.691026
60,0.682463
80,0.683127
100,0.687865
120,0.681235
140,0.685440


Saved to adapter_qwen


In [20]:
train_single_model("TinyLlama/TinyLlama-1.1B-Chat-v1.0","adapter_tinyllama")


===== TRAINING TinyLlama/TinyLlama-1.1B-Chat-v1.0 =====


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Token indices sequence length is longer than the specified maximum sequence length for this model (2154 > 2048). Running this sequence through the model will result in indexing errors


Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
20,0.692023
40,0.692661
60,0.694609
80,0.695468
100,0.692601
120,0.688915
140,0.686964


Saved to adapter_tinyllama


In [24]:
from huggingface_hub import login
login()
train_single_model("google/gemma-2b-it","adapter_gemma")


===== TRAINING google/gemma-2b-it =====


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' to 'google/gemma-2b-it'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Step,Training Loss
20,0.692704
40,0.694250
60,0.691587
80,0.691699
100,0.689469
120,0.685896
140,0.686850


Saved to adapter_gemma


In [25]:
model_info = [
    ("Qwen/Qwen2.5-1.5B-Instruct", "adapter_qwen"),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", "adapter_tinyllama"),
    ("google/gemma-2b-it", "adapter_gemma"),
]

In [26]:
def generate_responses(model, prompts, tokenizer, max_new_tokens=200):
    responses = []
    device = next(model.parameters()).device

    for i in tqdm(range(len(prompts))):
        formatted = tokenizer.apply_chat_template(
            prompts[i],
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            formatted,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        gen = output[:, inputs["input_ids"].shape[-1]:]
        text = tokenizer.decode(gen[0], skip_special_tokens=True)
        responses.append(text)

    return responses

In [27]:
def get_stats(texts):
    lengths = [len(t) for t in texts]
    unique_chars = [len(set(t)) for t in texts]

    return {
        "avg_len": float(np.mean(lengths)),
        "std_len": float(np.std(lengths)),
        "avg_unique_chars": float(np.mean(unique_chars)),
    }

In [28]:
results = {}

n_eval = 50
eval_subset = dataset_eval.select(range(n_eval))
prompts = [ex[:-1] for ex in eval_subset["chosen"]]

for model_name, adapter_path in model_info:

    print(f"\n===== EVALUATING {model_name} =====")

    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    base_model.eval()

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading DPO model...")
    dpo_model = PeftModel.from_pretrained(base_model, adapter_path)
    dpo_model.eval()

    print("Generating BASE responses...")
    base_responses = generate_responses(base_model, prompts, tokenizer)

    print("Generating DPO responses...")
    dpo_responses = generate_responses(dpo_model, prompts, tokenizer)

    # ===== статистика
    stats_base = get_stats(base_responses)
    stats_dpo  = get_stats(dpo_responses)

    results[model_name] = {
        "base": stats_base,
        "dpo": stats_dpo
    }

    del base_model
    del dpo_model
    torch.cuda.empty_cache()


===== EVALUATING Qwen/Qwen2.5-1.5B-Instruct =====
Loading base model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading DPO model...
Generating BASE responses...


100%|██████████| 50/50 [09:18<00:00, 11.17s/it]


...

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Loading DPO model...
Generating BASE responses...


100%|██████████| 50/50 [05:31<00:00,  6.62s/it]


Generating DPO responses...


100%|██████████| 50/50 [05:30<00:00,  6.61s/it]


In [29]:
print("\n===== FINAL RESULTS =====")
for model, stats in results.items():
    print(f"\n{model}")
    print("BASE:", stats["base"])
    print("DPO :", stats["dpo"])


===== FINAL RESULTS =====

Qwen/Qwen2.5-1.5B-Instruct
BASE: {'avg_len': 593.26, 'std_len': 415.53136151198026, 'avg_unique_chars': 38.2}
DPO : {'avg_len': 593.26, 'std_len': 415.53136151198026, 'avg_unique_chars': 38.2}

TinyLlama/TinyLlama-1.1B-Chat-v1.0
BASE: {'avg_len': 561.02, 'std_len': 262.8163229329564, 'avg_unique_chars': 38.12}
DPO : {'avg_len': 561.02, 'std_len': 262.8163229329564, 'avg_unique_chars': 38.12}

google/gemma-2b-it
BASE: {'avg_len': 492.12, 'std_len': 364.04690027522554, 'avg_unique_chars': 36.52}
DPO : {'avg_len': 492.12, 'std_len': 364.04690027522554, 'avg_unique_chars': 36.52}


In [31]:
from IPython.display import display, HTML

N_EXAMPLES = 5

for model_name, adapter_path in model_info:

    print(f"\n===== {model_name} =====")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    prompts_subset = prompts[:N_EXAMPLES]

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    base_model.eval()
    dpo_model = PeftModel.from_pretrained(base_model, adapter_path)
    dpo_model.eval()

    base_responses = generate_responses(base_model, prompts_subset, tokenizer)
    dpo_responses  = generate_responses(dpo_model, prompts_subset, tokenizer)

    for i in range(N_EXAMPLES):
        prompt_text = "\n".join(f"[{m['role']}] {m['content']}" for m in prompts_subset[i])
        base_text   = base_responses[i]
        dpo_text    = dpo_responses[i]

        html = f"""
        <div style="border:1px solid #ccc; padding:10px; margin:10px 0; max-width:900px;">
            <b>Example {i+1}</b><br><br>
            <b style="color:#555;">Prompt:</b>
            <div style="background:#f8f8f8; padding:6px;">{prompt_text}</div><br>
            <b style="color:#e88;">BASE:</b>
            <div style="background:#f0e0e0; padding:6px;">{base_text}</div><br>
            <b style="color:#48c;">DPO:</b>
            <div style="background:#e0f0ff; padding:6px;">{dpo_text}</div>
        </div>
        """
        display(HTML(html))

    del base_model
    del dpo_model
    torch.cuda.empty_cache()


===== Qwen/Qwen2.5-1.5B-Instruct =====


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

100%|██████████| 5/5 [01:03<00:00, 12.78s/it]



===== TinyLlama/TinyLlama-1.1B-Chat-v1.0 =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:33<00:00,  6.69s/it]



===== google/gemma-2b-it =====


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:29<00:00,  5.96s/it]
